# «Тишина АЗ-26» — решение (v5: признаки под «дрейфовый» тип эпизодов, оценка на нём)

**Идея.** Предотказные эпизоды (~20 точек, ~4.7% строк) почти не видны по уровню отдельных датчиков:
значения остаются в штатных границах. Но внутри эпизода у многих датчиков одновременно
растёт *локальный шум* (дисперсия первой разности), дисперсия уровня в окне относительно окружения,
а в конце эпизода часто бывает «возврат» скачком. Сигнал размазан по всем 15 датчикам, поэтому:

1. для каждого датчика в сеансе считаем оконные статистики (центрированные окна — разметка ретроспективная,
   поэтому можно смотреть и в прошлое, и в будущее);
2. агрегируем их **по датчикам** (mean / max / q80 / top-3 / счётчики) — это устойчивее, чем 400+ посенсорных признаков;
3. добавляем ранги агрегатов внутри сеанса (нормировка между сеансами) и признаки положения относительно смены режима;
4. LightGBM с сильной регуляризацией, валидация GroupKFold по `session_id`;
5. сглаживаем предсказание скользящим средним по времени внутри сеанса (эпизоды — непрерывные отрезки).

In [1]:
import numpy as np, pandas as pd, lightgbm as lgb, warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
warnings.filterwarnings('ignore')
print('lightgbm', lgb.__version__, '| pandas', pd.__version__)

train = pd.read_csv('telemetry_train.csv')
test = pd.read_csv('telemetry_test.csv')
print(train.shape, test.shape, 'доля аномалий:', round(train.is_anomaly.mean(), 4))
print('сеансов train/test:', train.session_id.nunique(), test.session_id.nunique())

lightgbm 4.7.0 | pandas 3.0.6


(120000, 20) (40000, 19) доля аномалий: 0.0475
сеансов train/test: 32 16


## Признаки

In [2]:
import numpy as np, pandas as pd

SENS = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
        'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']
HW = (4, 8, 12, 20)   # полуширины «внутреннего» окна
CTX = 30              # длина контекста слева и справа


def _wsum(v, lo, hi):
    """Сумма v[i+lo : i+hi] для каждого i (NaN -> 0), окна обрезаются по краям."""
    n = len(v)
    cs = np.r_[0, np.cumsum(v)]
    i = np.arange(n)
    a = np.clip(i + lo, 0, n); b = np.clip(i + hi, 0, n)
    return cs[b] - cs[a]


def _wmean(x, lo, hi):
    m = ~np.isnan(x)
    s = _wsum(np.where(m, x, 0), lo, hi); c = _wsum(m.astype(float), lo, hi)
    s2 = _wsum(np.where(m, x * x, 0), lo, hi)
    mu = s / np.maximum(c, 1)
    var = np.maximum(s2 / np.maximum(c, 1) - mu ** 2, 0)
    mu[c == 0] = np.nan
    return mu, var, c


def session_feats(d):
    d = d.reset_index(drop=True)
    n = len(d); pos = np.arange(n)
    F = {}
    F['mode'] = d['mode'].map({'cruise': 0, 'maneuver': 1, 'eclipse': 2}).values
    ch = np.r_[0, np.where(d['mode'].values[1:] != d['mode'].values[:-1])[0] + 1, n]
    k = np.searchsorted(ch, pos, side='right')
    F['dist_prev_mode'] = pos - ch[k - 1]
    F['dist_next_mode'] = ch[np.minimum(k, len(ch) - 1)] - pos
    F['rel_pos'] = pos / n

    agg = {}
    for s in SENS:
        raw = d[s].values.astype(float)
        dx = np.diff(raw, prepend=np.nan)
        mad = np.nanmedian(np.abs(dx - np.nanmedian(dx)))
        ns = 1.4826 * mad + 1e-12                 # шум по первой разности
        z = raw / ns
        adz = np.abs(dx / ns)
        # скачки: максимум |dz| в окнах вперёд/назад
        a0 = np.nan_to_num(adz)
        sa = pd.Series(a0)
        for w in (10, 25):
            f = sa[::-1].rolling(w, min_periods=1).max()[::-1].shift(-1).values
            b = sa.rolling(w, min_periods=1).max().values
            F[f'{s}_jmpf{w}'] = f; F[f'{s}_jmpb{w}'] = b
        F[f'{s}_vol11'] = pd.Series(adz).rolling(11, center=True, min_periods=1).mean().values
        # «горб»: среднее внутри окна против контекста слева+справа
        for h in HW:
            mi, vi, ci = _wmean(z, -h, h + 1)
            ml, vl, cl = _wmean(z, -h - CTX, -h)
            mr, vr, cr = _wmean(z, h + 1, h + 1 + CTX)
            mc = np.where(np.isnan(ml), mr, np.where(np.isnan(mr), ml, (ml * cl + mr * cr) / np.maximum(cl + cr, 1)))
            vc = (vl * cl + vr * cr) / np.maximum(cl + cr, 1) + 1e-9
            bump = (mi - mc) / np.sqrt(vc) * np.sqrt(np.maximum(ci, 1))
            ratio = np.sqrt(vi / vc)
            lr = (mr - ml) / np.sqrt(vc)
            F[f'{s}_bump{h}'] = bump
            F[f'{s}_vr{h}'] = ratio
            F[f'{s}_lr{h}'] = lr
            agg.setdefault(f'bump{h}', []).append(np.abs(np.nan_to_num(bump)))
            agg.setdefault(f'vr{h}', []).append(np.nan_to_num(ratio, nan=1))
        # уровень шума: дисперсия первой разности в окне против длинного контекста
        e2 = (dx / ns) ** 2
        for h in (6, 10, 16):
            m, _, _ = _wmean(e2, -h, h + 1)
            for C in (60, 150):
                mc, _, _ = _wmean(e2, -h - C, h + C + 1)
                v = np.log(m + 1e-3) - np.log(mc + 1e-3)
                F[f'{s}_dvr{h}_{C}'] = v
                agg.setdefault(f'dvr{h}_{C}', []).append(np.nan_to_num(v))
        # дисперсия уровня в окне против контекста 80
        for h in (6, 10, 14):
            mi, vi, ci = _wmean(z, -h, h + 1)
            ml, vl, cl = _wmean(z, -h - 80, -h)
            mr, vr_, cr = _wmean(z, h + 1, h + 81)
            vc = (vl * cl + vr_ * cr) / np.maximum(cl + cr, 1) + 1e-9
            v = np.log(vi + 1e-9) - np.log(vc)
            F[f'{s}_lvr{h}'] = v
            agg.setdefault(f'lvr{h}', []).append(np.nan_to_num(v))
        med = pd.Series(z).rolling(151, center=True, min_periods=1).median().values
        sm = pd.Series(z).rolling(9, center=True, min_periods=1).mean().values
        F[f'{s}_dev151'] = sm - med
        agg.setdefault('dev151', []).append(np.abs(np.nan_to_num(F[f'{s}_dev151'])))
        for w in (10, 25):
            agg.setdefault(f'jmpf{w}', []).append(F[f'{s}_jmpf{w}'])
            agg.setdefault(f'jmpb{w}', []).append(F[f'{s}_jmpb{w}'])
        agg.setdefault('lr8', []).append(np.abs(np.nan_to_num(F[f'{s}_lr8'])))
        agg.setdefault('vol11', []).append(np.nan_to_num(F[f'{s}_vol11']))
    for k_, v in agg.items():
        A = np.vstack(v)
        F[f'agg_{k_}_mean'] = A.mean(0)
        F[f'agg_{k_}_max'] = A.max(0)
        F[f'agg_{k_}_q80'] = np.quantile(A, 0.8, axis=0)
        F[f'agg_{k_}_top3'] = np.sort(A, 0)[-3:].mean(0)
        if k_.startswith(('dvr', 'lvr')):
            F[f'agg_{k_}_n05'] = (A > 0.5).sum(0)
            F[f'agg_{k_}_n10'] = (A > 1.0).sum(0)
    out = pd.DataFrame(F)
    out['timestamp_id'] = d['timestamp_id'].values
    out['session_id'] = d['session_id'].values
    return out


def build(df):
    return pd.concat([session_feats(g) for _, g in df.groupby('session_id', sort=False)], ignore_index=True)


### Маркеры границ эпизодов (v2)
Разбор ошибок показал, что границы эпизодов резкие: в первой строке эпизода и в первой штатной строке после него
часто скачут **сразу несколько датчиков** (≥2 датчика с |Δ|>4σ: 2.8% случайных строк против 16% начал и 25% концов).
Добавляем счётчики одновременных скачков, их скользящие средние и расстояния до ближайших таких строк.

In [3]:

SENS_ = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
        'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']
EDGES = np.array([0, .25, .5, .75, 1, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 11, 15, 25, np.inf])


def norm_diffs(df, per_session=True):
    """|первая разность| / робастный масштаб шума; NaN там, где разность не определена."""
    g = df.session_id.values
    X = df[SENS_].values.astype(float)
    D = np.diff(X, axis=0, prepend=np.nan)
    D[np.r_[True, g[1:] != g[:-1]]] = np.nan
    Z = np.empty_like(D)
    for s in np.unique(g):
        m = g == s
        d = D[m]
        ns = 1.4826 * np.nanmedian(np.abs(d - np.nanmedian(d, 0)), 0) + 1e-12
        Z[m] = np.abs(d / ns)
    return Z




def bracket_feats(df):
    """Признаки одновременных скачков нескольких датчиков (маркеры границ эпизода)."""
    Z = np.nan_to_num(norm_diffs(df))
    g = df.session_id.values
    F = {}
    for thr in (3, 4, 6):
        F[f'cnt{thr}'] = (Z > thr).sum(1).astype(float)
    F['sumlog'] = np.log1p(Z).sum(1)
    out = pd.DataFrame(F)
    res = {}
    for s in pd.unique(g):
        m = np.where(g == s)[0]; n = len(m); pos = np.arange(n)
        R = {}
        for thr, k in ((3, 2), (4, 2), (4, 1), (6, 1)):
            ev = out[f'cnt{thr}'].values[m] >= k
            idx = np.where(ev)[0]
            j = np.searchsorted(idx, pos, side='right')
            prev = np.where(j > 0, idx[np.maximum(j - 1, 0)], -10 ** 4)       # событие на строке <= t (начало)
            nxt = np.where(j < len(idx), idx[np.minimum(j, len(idx) - 1)], 10 ** 4 + n)  # событие > t (конец)
            dp = np.minimum(pos - prev, 200); dn = np.minimum(nxt - pos, 200); span = np.minimum(nxt - prev, 400)
            key = f'{thr}_{k}'
            R[f'dprev_{key}'] = dp; R[f'dnext_{key}'] = dn; R[f'span_{key}'] = span
            R[f'brk_{key}'] = ((span >= 6) & (span <= 60)).astype(float)
        for c in ('cnt3', 'cnt4', 'sumlog'):
            v = pd.Series(out[c].values[m])
            for w in (15, 31):
                R[f'{c}_m{w}'] = v.rolling(w, center=True, min_periods=1).mean().values
        res[s] = pd.DataFrame(R, index=m)
    return pd.concat([out, pd.concat(res.values()).sort_index()], axis=1)


In [4]:
from sklearn.linear_model import Ridge

def diff_residuals(df):
    """Первые разности каждого датчика минус их линейный прогноз по разностям остальных
    датчиков (лаги -1, 0, +1). Регрессия обучается без меток — на всех строках df."""
    X = df.groupby('session_id')[SENS].transform(lambda v: v.interpolate(limit_direction='both'))
    g = df.session_id.values
    D = X.groupby(g).diff().fillna(0); D = D / D.std()
    lags = [D.groupby(g).shift(k).fillna(0).add_suffix(f'_{k}') for k in (-1, 1)]
    R = {}
    for s in SENS:
        oth = [o for o in SENS if o != s]
        Z = pd.concat([D[oth]] + [l[[f'{o}_{k}' for o in oth]] for l, k in zip(lags, (-1, 1))], axis=1)
        e = D[s] - Ridge(1.0).fit(Z, D[s]).predict(Z)
        R[s] = np.where(df[s].isna() | df[s].groupby(g).shift(1).isna(), np.nan, e)
    return pd.DataFrame(R, index=df.index)


def _channel(F, agg, name, dz, lvl):
    """dz — нормированные первые разности, lvl — нормированный уровень."""
    adz = np.abs(dz); sa = pd.Series(np.nan_to_num(adz))
    for w in (10, 25):
        agg.setdefault(f'{name}jmpf{w}', []).append(sa[::-1].rolling(w, min_periods=1).max()[::-1].shift(-1).fillna(0).values)
        agg.setdefault(f'{name}jmpb{w}', []).append(sa.rolling(w, min_periods=1).max().values)
    agg.setdefault(f'{name}vol11', []).append(np.nan_to_num(pd.Series(adz).rolling(11, center=True, min_periods=1).mean().values))
    e2 = dz ** 2
    for h in (6, 10, 16):
        m, _, _ = _wmean(e2, -h, h + 1)
        for C in (60, 150):
            mc, _, _ = _wmean(e2, -h - C, h + C + 1)
            agg.setdefault(f'{name}dvr{h}_{C}', []).append(np.nan_to_num(np.log(m + 1e-3) - np.log(mc + 1e-3)))
    for h in (4, 8, 12, 20):
        mi, vi, ci = _wmean(lvl, -h, h + 1)
        ml, vl, cl = _wmean(lvl, -h - 30, -h); mr, vr, cr = _wmean(lvl, h + 1, h + 31)
        mc = np.where(np.isnan(ml), mr, np.where(np.isnan(mr), ml, (ml * cl + mr * cr) / np.maximum(cl + cr, 1)))
        vc = (vl * cl + vr * cr) / np.maximum(cl + cr, 1) + 1e-9
        agg.setdefault(f'{name}bump{h}', []).append(np.abs(np.nan_to_num((mi - mc) / np.sqrt(vc) * np.sqrt(np.maximum(ci, 1)))))
        agg.setdefault(f'{name}vr{h}', []).append(np.nan_to_num(np.sqrt(vi / vc), nan=1))
        if h == 8:
            agg.setdefault(f'{name}lr8', []).append(np.abs(np.nan_to_num((mr - ml) / np.sqrt(vc))))
    for h in (6, 10, 14):
        mi, vi, ci = _wmean(lvl, -h, h + 1)
        ml, vl, cl = _wmean(lvl, -h - 80, -h); mr, vr, cr = _wmean(lvl, h + 1, h + 81)
        vc = (vl * cl + vr * cr) / np.maximum(cl + cr, 1) + 1e-9
        agg.setdefault(f'{name}lvr{h}', []).append(np.nan_to_num(np.log(vi + 1e-9) - np.log(vc)))
    med = pd.Series(lvl).rolling(151, center=True, min_periods=1).median().values
    sm = pd.Series(lvl).rolling(9, center=True, min_periods=1).mean().values
    agg.setdefault(f'{name}dev151', []).append(np.abs(np.nan_to_num(sm - med)))


### Физические остатки (v4)
Для «энергетических» и тепловых датчиков строим линейную модель уровня по экспоненциально сглаженным
(постоянные времени 3–100 точек) значениям остальных датчиков и режима — грубая модель тепловой инерции
и связи ток солнечных батарей ↔ световой поток (R² до 0.97). Модель обучается без меток на train+test.
На остатках считаем те же оконные признаки, агрегируем по датчикам и ранжируем внутри сеанса.

In [5]:

PS = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator', 'p_fuel', 'signal_strength']


def phys_residuals(df):
    """Остаток уровня датчика относительно линейной модели от экспоненциально сглаженных (причинно)
    значений остальных датчиков и режима. Средние по сеансу вычитаются. Без меток."""
    g = df.session_id.values
    X = df.groupby('session_id')[PS].transform(lambda v: v.interpolate(limit_direction='both'))
    M = pd.get_dummies(df['mode']).astype(float)
    drv = {}
    for c in PS + list(M.columns):
        base = pd.Series(X[c].values if c in PS else M[c].values)
        for tau in (3, 10, 30, 100):
            drv[f'{c}_e{tau}'] = base.groupby(g).transform(lambda s: s.ewm(span=tau, adjust=False).mean()).values
    D = pd.DataFrame(drv)
    D = D - D.groupby(g).transform('mean')
    R = {}
    for t in PS:
        cols = [c for c in D.columns if not c.startswith(t + '_')]
        tt = X[t] - X[t].groupby(g).transform('mean')
        e = tt.values - Ridge(1.0).fit(D[cols], tt).predict(D[cols])
        R[t] = np.where(df[t].isna(), np.nan, e / np.std(e))
    return pd.DataFrame(R, index=df.index)


both = pd.concat([train.drop(columns='is_anomaly'), test], ignore_index=True)
R_phys = phys_residuals(both)

def phys_feats(df, R):
    g = df.session_id.values; R = R.reset_index(drop=True); rows = []
    for s in pd.unique(g):
        m = np.where(g == s)[0]; agg = {}
        for c in PS:
            e = R[c].values[m]; dz = np.diff(e, prepend=np.nan)
            dsc = 1.4826 * np.nanmedian(np.abs(dz - np.nanmedian(dz))) + 1e-12
            _channel({}, agg, 'p_', dz / dsc, e / dsc)
        out = {}
        for k, v in agg.items():
            A = np.vstack(v)
            out[f'agg_{k}_mean'] = A.mean(0); out[f'agg_{k}_max'] = A.max(0); out[f'agg_{k}_top3'] = np.sort(A, 0)[-3:].mean(0)
        rows.append(pd.DataFrame(out, index=m))
    P_ = pd.concat(rows).sort_index()
    return pd.concat([P_, P_.groupby(g).rank(pct=True).add_suffix('_rk')], axis=1)

P_tr = phys_feats(train, R_phys.iloc[:len(train)])
P_te = phys_feats(test, R_phys.iloc[len(train):])
print('физических признаков:', P_tr.shape[1])

физических признаков: 144


### Признаки «дрейфового» типа (v5)
Разбор показал два механизма эпизодов: (1) у 3–4 датчиков шум возрастает в 4–14 раз — их ~24% в train;
(2) у части датчиков уровень плавно уходит и в конце эпизода возвращается скачком — остальные ~76%.
В тесте эпизодов первого типа почти нет (одновременных всплесков шума ~в 10 раз меньше, чем в train),
поэтому качество модели меряем по AUC **на эпизодах второго типа** и добавляем признаки под него:
остатки разностей после регрессии на остальные датчики, сканирование формы «рампа→сброс»,
дисперсия уровня в окне против соседних окон той же длины.

In [6]:
def session_feats2(d, res):
    n = len(d); pos = np.arange(n); F = {}
    F['mode'] = d['mode'].map({'cruise': 0, 'maneuver': 1, 'eclipse': 2}).values
    ch = np.r_[0, np.where(d['mode'].values[1:] != d['mode'].values[:-1])[0] + 1, n]
    k = np.searchsorted(ch, pos, side='right')
    F['dist_prev_mode'] = pos - ch[k - 1]
    F['dist_next_mode'] = ch[np.minimum(k, len(ch) - 1)] - pos
    F['rel_pos'] = pos / n
    agg = {}
    for s in SENS:
        raw = d[s].values.astype(float)
        dx = np.diff(raw, prepend=np.nan)
        ns = 1.4826 * np.nanmedian(np.abs(dx - np.nanmedian(dx))) + 1e-12
        _channel(F, agg, '', dx / ns, raw / ns)
        e = res[s].values
        es = 1.4826 * np.nanmedian(np.abs(e - np.nanmedian(e))) + 1e-12
        e = e / es
        _channel(F, agg, 'r_', e, np.nancumsum(np.nan_to_num(e)))   # уровень = накопленный остаток
    for k_, v in agg.items():
        A = np.vstack(v)
        F[f'agg_{k_}_mean'] = A.mean(0); F[f'agg_{k_}_max'] = A.max(0)
        F[f'agg_{k_}_q80'] = np.quantile(A, 0.8, axis=0)
        F[f'agg_{k_}_top3'] = np.sort(A, 0)[-3:].mean(0)
        if 'dvr' in k_ or 'lvr' in k_:
            F[f'agg_{k_}_n05'] = (A > 0.5).sum(0); F[f'agg_{k_}_n10'] = (A > 1.0).sum(0)
    out = pd.DataFrame(F)
    out['timestamp_id'] = d['timestamp_id'].values; out['session_id'] = d['session_id'].values
    return out


def build2(df, res):
    return pd.concat([session_feats2(g.reset_index(drop=True), res.loc[g.index].reset_index(drop=True))
                      for _, g in df.groupby('session_id', sort=False)], ignore_index=True)



RS = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
      'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']
LENS = (8, 12, 16, 20, 26, 34, 44)


def _cs(v):
    return np.r_[0.0, np.cumsum(v)]


def ramp_scan(x, lens=LENS, m=10, shape='ramp'):
    """Для каждого конца e (первая строка после окна) и длины L: МНК-амплитуда добавки формы w
    в окне [e-L, e) относительно линии между средними до и после окна. Возвращает dict L -> z[e]
    (робастно нормированная по сеансу)."""
    n = len(x); c = _cs(x); i = np.arange(x.size); ci = _cs(x * i)
    out = {}
    for L in lens:
        e = np.arange(n + 1)
        s = e - L
        ok = (s - m >= 0) & (e + m <= n)
        z = np.full(n + 1, np.nan)
        ee, ss = e[ok], s[ok]
        pre = (c[ss] - c[ss - m]) / m; post = (c[ee + m] - c[ee]) / m
        # сумма x и сумма x*t по окну
        Sx = c[ee] - c[ss]; Sxt = ci[ee] - ci[ss]
        k = np.arange(1, L + 1, dtype=float)               # позиции 1..L внутри окна
        if shape == 'ramp':
            w = k / L
        else:
            w = np.ones(L)
        # база b_t = pre + (post-pre)*(k-0.5)/L ; d_t = x_t - b_t
        bw = (k - 0.5) / L
        Sw = w.sum(); Sww = (w * w).sum(); Swb = (w * bw).sum()
        # sum w*x: w линейна по k для ramp -> через Sx и Sxt (t = ss + k - 1)
        if shape == 'ramp':
            Swx = (Sxt - (ss - 1) * Sx) / L
        else:
            Swx = Sx
        Swd = Swx - pre * Sw - (post - pre) * Swb
        z[ok] = Swd / np.sqrt(Sww)
        zz = z[:n + 1]
        med = np.nanmedian(zz); sc = 1.4826 * np.nanmedian(np.abs(zz - med)) + 1e-12
        out[L] = (zz - med) / sc
    return out


def row_cover_max(zdict, n):
    """Строковый признак: максимум |z| по окнам [e-L, e), покрывающим строку t."""
    best = np.zeros(n); bestL = np.zeros(n)
    for L, z in zdict.items():
        a = np.nan_to_num(np.abs(z[1:]), nan=0.0)        # индекс e-1 -> e=1..n
        # строка t покрыта при e в (t, t+L]  ->  a[e-1] при e-1 в [t, t+L-1]
        r = pd.Series(a[::-1]).rolling(L, min_periods=1).max().values[::-1]
        upd = r > best; best = np.where(upd, r, best); bestL = np.where(upd, L, bestL)
    return best, bestL


def ramp_feats(df):
    g = df.session_id.values; F = {}
    parts = []
    for s in pd.unique(g):
        m = np.where(g == s)[0]; n = len(m); A = []; Bx = []
        for c in RS:
            x = df[c].iloc[m].interpolate(limit_direction='both').values.astype(float)
            dx = np.diff(x); x = x / (1.4826 * np.median(np.abs(dx - np.median(dx))) + 1e-12)
            zr, _ = row_cover_max(ramp_scan(x, shape='ramp'), n)
            zb, _ = row_cover_max(ramp_scan(x, shape='box'), n)
            A.append(zr); Bx.append(zb)
        A = np.vstack(A); Bx = np.vstack(Bx)
        P = {}
        for nm, M in (('ramp', A), ('box', Bx)):
            P[f'{nm}_max'] = M.max(0); P[f'{nm}_top3'] = np.sort(M, 0)[-3:].mean(0)
            P[f'{nm}_mean'] = M.mean(0); P[f'{nm}_n3'] = (M > 3).sum(0); P[f'{nm}_n4'] = (M > 4).sum(0)
            for j, c in enumerate(RS):
                P[f'{nm}_{c}'] = M[j]
        parts.append(pd.DataFrame(P, index=m))
    out = pd.concat(parts).sort_index()
    rk = out[[c for c in out.columns if c.endswith(('_max', '_top3', '_mean', '_n3', '_n4'))]].groupby(g).rank(pct=True).add_suffix('_rk')
    return pd.concat([out, rk], axis=1)



AS = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
      'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']


def adj_feats(df, hs=(5, 8, 12, 16, 22)):
    """Дисперсия уровня в центрированном окне против соседних окон той же длины слева и справа
    (лучший признак «дрейфового» типа эпизодов), плюс несимметричные варианты (окно, кончающееся/
    начинающееся в t). Агрегаты по датчикам."""
    g = df.session_id.values; parts = []
    for s in pd.unique(g):
        m = np.where(g == s)[0]; agg = {}
        for c in AS:
            x = df[c].iloc[m].values.astype(float)
            for h in hs:
                L = 2 * h + 1
                mi, vi, ci = _wmean(x, -h, h + 1)
                _, vl, cl = _wmean(x, -h - L, -h); _, vr, cr = _wmean(x, h + 1, h + 1 + L)
                vadj = np.where((cl > 3) & (cr > 3), 0.5 * (vl + vr), np.where(cl > 3, vl, vr))
                agg.setdefault(f'adj{h}', []).append(np.nan_to_num(np.log(vi + 1e-12) - np.log(vadj + 1e-12)))
                # окно, заканчивающееся в t (эпизод справа от t закончился?) и начинающееся в t
                _, vb, cb = _wmean(x, -L + 1, 1); _, vbb, _ = _wmean(x, -2 * L + 1, -L + 1)
                _, vf, cf = _wmean(x, 0, L); _, vff, _ = _wmean(x, L, 2 * L)
                agg.setdefault(f'adjb{h}', []).append(np.nan_to_num(np.log(vb + 1e-12) - np.log(vbb + 1e-12)))
                agg.setdefault(f'adjf{h}', []).append(np.nan_to_num(np.log(vf + 1e-12) - np.log(vff + 1e-12)))
        P = {}
        for k, v in agg.items():
            A = np.clip(np.vstack(v), -8, 8)
            P[f'{k}_max'] = A.max(0); P[f'{k}_top3'] = np.sort(A, 0)[-3:].mean(0); P[f'{k}_mean'] = A.mean(0)
            P[f'{k}_n1'] = (A > 1).sum(0)
        parts.append(pd.DataFrame(P, index=m))
    out = pd.concat(parts).sort_index()
    return pd.concat([out, out.groupby(g).rank(pct=True).add_suffix('_rk')], axis=1)


def resid_group(X, sess):
    A = [c for c in X.columns if c.startswith('agg_r_')]
    return pd.concat([X[A], X[A].groupby(sess).rank(pct=True).add_suffix('_rk')], axis=1).reset_index(drop=True)

def ramp_group(df):
    Rf = ramp_feats(df).reset_index(drop=True)
    return Rf[[c for c in Rf.columns if c.endswith(('_max', '_top3', '_mean', '_n3', '_n4', '_rk'))]]

if 'both' not in globals():
    both = pd.concat([train.drop(columns='is_anomaly'), test], ignore_index=True)
res_all = diff_residuals(both)
Q_tr = pd.concat([resid_group(build2(train, res_all.iloc[:len(train)].set_index(train.index)), train.session_id.values),
                  ramp_group(train), adj_feats(train).reset_index(drop=True)], axis=1)
Q_te = pd.concat([resid_group(build2(test, res_all.iloc[len(train):].set_index(test.index)), test.session_id.values),
                  ramp_group(test), adj_feats(test).reset_index(drop=True)], axis=1)
print('новых признаков:', Q_tr.shape[1])

новых признаков: 368


In [7]:
Xtr = build(train)
Xte = build(test)
assert (Xtr.timestamp_id.values == train.timestamp_id.values).all()
assert (Xte.timestamp_id.values == test.timestamp_id.values).all()

BASE = ['mode', 'dist_prev_mode', 'dist_next_mode', 'rel_pos']
AGG = [c for c in Xtr.columns if c.startswith('agg')]

def make_matrix(X):
    R = X[AGG].groupby(X.session_id.values).rank(pct=True).add_suffix('_rk')   # ранг внутри сеанса
    return pd.concat([X[AGG + BASE], R], axis=1)

Btr = bracket_feats(train).reset_index(drop=True)
Bte = bracket_feats(test).reset_index(drop=True)

XX_tr = pd.concat([make_matrix(Xtr), Btr, P_tr, Q_tr], axis=1)
XX_te = pd.concat([make_matrix(Xte), Bte, P_te, Q_te], axis=1)
y, groups = train.is_anomaly.values, train.session_id.values
print('признаков:', XX_tr.shape[1])

признаков: 770


## Валидация (GroupKFold по сеансам)

In [8]:
PARAMS = dict(objective='binary', learning_rate=0.02, num_leaves=15, min_child_samples=400,
              feature_fraction=0.2, bagging_fraction=0.7, bagging_freq=1, lambda_l2=30,
              verbose=-1, n_jobs=4)
ROUNDS, SMOOTH, SEEDS = 400, 15, [0, 1, 2, 3, 4]

def smooth(p, sess, w=SMOOTH):
    return pd.Series(p).groupby(np.asarray(sess)).transform(
        lambda v: v.rolling(w, center=True, min_periods=1).mean()).values

oof = np.zeros(len(y))
for k, (a, b) in enumerate(GroupKFold(8).split(XX_tr, y, groups)):
    m = lgb.train(dict(PARAMS, seed=0), lgb.Dataset(XX_tr.iloc[a], y[a]), ROUNDS)
    oof[b] = m.predict(XX_tr.iloc[b])
    print(f'fold {k}: AUC={roc_auc_score(y[b], oof[b]):.4f}  (сглаж. {roc_auc_score(y[b], smooth(oof[b], groups[b])):.4f})')
print(f'OOF ROC-AUC: {roc_auc_score(y, oof):.4f}')
print(f'OOF ROC-AUC после сглаживания (окно {SMOOTH}): {roc_auc_score(y, smooth(oof, groups)):.4f}')

def finalize(p, sess):
    # сглаживание + ранг внутри сеанса (убирает сдвиг уровня оценок между сеансами)
    return pd.Series(smooth(p, sess)).groupby(np.asarray(sess)).rank(pct=True).values
print(f'OOF ROC-AUC после сглаживания и ранга в сеансе: {roc_auc_score(y, finalize(oof, groups)):.4f}')

# эпизоды «шумового» типа (в тесте их почти нет) — исключаем из оценки, чтобы метрика была похожа на тест
Xr_ = train[SENS_].values.astype(float); D_ = np.diff(Xr_, axis=0, prepend=np.nan)
D_[np.r_[True, groups[1:] != groups[:-1]]] = np.nan
r_ = np.diff(np.r_[0, y, 0]); st_, en_ = np.where(r_ == 1)[0], np.where(r_ == -1)[0]
noise_ep = np.zeros(len(y), bool)
for s_, e_ in zip(st_, en_):
    ctx = np.vstack([D_[max(s_ - 60, 0):s_], D_[e_ + 1:e_ + 61]])
    if np.nanmax(np.nanstd(D_[s_ + 1:e_], 0) / (np.nanstd(ctx, 0) + 1e-12)) >= 2.5: noise_ep[s_:e_] = True
ev = ~noise_ep
print(f'эпизодов шумового типа: {len(set(np.cumsum(r_[:-1]==1)[noise_ep]))} из {len(st_)}')
print(f'OOF ROC-AUC на эпизодах дрейфового типа (похоже на тест): {roc_auc_score(y[ev], finalize(oof, groups)[ev]):.4f}')

fold 0: AUC=0.7135  (сглаж. 0.7190)


fold 1: AUC=0.6936  (сглаж. 0.7020)


fold 2: AUC=0.8132  (сглаж. 0.8183)


fold 3: AUC=0.6828  (сглаж. 0.6819)


fold 4: AUC=0.7057  (сглаж. 0.7038)


fold 5: AUC=0.7301  (сглаж. 0.7396)


fold 6: AUC=0.7009  (сглаж. 0.7183)


fold 7: AUC=0.7375  (сглаж. 0.7396)
OOF ROC-AUC: 0.7196
OOF ROC-AUC после сглаживания (окно 15): 0.7247


OOF ROC-AUC после сглаживания и ранга в сеансе: 0.7260
эпизодов шумового типа: 66 из 277
OOF ROC-AUC на эпизодах дрейфового типа (похоже на тест): 0.6531


## Обучение на всей выборке и сабмит

In [9]:
pred = np.zeros(len(XX_te))
for s in SEEDS:
    m = lgb.train(dict(PARAMS, seed=s), lgb.Dataset(XX_tr, y), ROUNDS)
    pred += m.predict(XX_te) / len(SEEDS)
pred = finalize(pred, test.session_id.values)

sub = pd.DataFrame({'timestamp_id': test.timestamp_id, 'anomaly_prob': pred})
sub.to_csv('submission_v5.csv', index=False)
print(sub.shape, sub.anomaly_prob.describe().round(4).to_dict())
sub.head()

(40000, 2) {'count': 40000.0, 'mean': 0.5002, 'std': 0.2887, 'min': 0.0003, '25%': 0.2502, '50%': 0.5002, '75%': 0.7502, 'max': 1.0}


,timestamp_id,anomaly_prob
0,220001,0.492922
1,220002,0.494944
2,220003,0.478766
3,220004,0.507583
4,220005,0.526289


In [10]:
imp = pd.Series(m.feature_importance('gain'), XX_tr.columns).sort_values(ascending=False)
imp.head(20).round(1)

agg_r_dvr6_150_max     8921.1
span_3_2               8207.2
agg_r_dvr6_150_top3    8124.6
agg_dvr10_60_max_rk    6690.1
span_6_1               5953.8
agg_r_jmpf25_top3      5867.8
rel_pos                5571.1
agg_lr8_q80            5190.6
span_4_2               5110.2
agg_r_jmpf10_mean      5091.2
dist_next_mode         4715.1
agg_r_jmpf25_max_rk    4395.8
agg_lr8_mean_rk        4249.9
agg_dvr6_150_max       4227.4
agg_r_jmpf10_top3      4110.4
agg_dvr10_60_n10_rk    4043.7
ramp_max_rk            4029.8
agg_dvr6_150_top3      3965.1
span_4_1               3948.9
agg_r_jmpb25_max_rk    3923.2
dtype: float64